# DeepFace Online Endpoint Deployment (Azure ML)

## Readfirst
This notebook deploys (or updates) a **Managed Online Endpoint** in Azure Machine Learning for DeepFace scoring.
It is **idempotent**: it checks whether assets already exist (model / environment / endpoint / deployment) and only creates them when missing.

### Prerequisites
- Install deps (must include `azure-ai-ml` and `azure-identity`).
- Authenticate with Azure: set `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET` in .dotenv
- Set workspace env vars in .dotenv: `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZUREML_WORKSPACE_NAME`.

### How to retrieve scoring URI + secret
Run all cells in this notebook. The cell in section "Get scoring URI and key" will print the scoring URI and secret key to use when calling the endpoint.

### Delete the endpoint
You have to delete the endpoint manually with the Azure ML SDK after you are done testing it, to avoid incurring ongoing costs. Setting up the CONFIRM_DELETE variable to True and running the last cell in this notebook will delete the endpoint.


# 1) Imports and configuration

In [1]:
import os
import json
import base64
import urllib.request
from pathlib import Path
from dotenv import load_dotenv, find_dotenv, set_key
from huggingface_hub import snapshot_download

from azure.identity import ClientSecretCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Environment,
    CodeConfiguration,
    OnlineRequestSettings,
    Model,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.core.polling import LROPoller

# Load environment variables from .env file
load_dotenv()

e:\Repository\FinalProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Authentication 
TENANT_ID = os.getenv("AZURE_TENANT_ID")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET")
if not all([TENANT_ID, CLIENT_ID, CLIENT_SECRET]):
    raise EnvironmentError("Missing required env vars for authentication: AZURE_TENANT_ID, AZURE_CLIENT_ID, AZURE_CLIENT_SECRET")
TENANT_ID_GPU = os.getenv("AZURE_TENANT_ID_GPU")
CLIENT_ID_GPU = os.getenv("AZURE_CLIENT_ID_GPU")
CLIENT_SECRET_GPU = os.getenv("AZURE_CLIENT_SECRET_GPU")
if not all([TENANT_ID_GPU, CLIENT_ID_GPU, CLIENT_SECRET_GPU]):
    raise EnvironmentError("Missing required env vars for GPU authentication: AZURE_TENANT_ID_GPU, AZURE_CLIENT_ID_GPU, AZURE_CLIENT_SECRET_GPU")

# Required workspace targeting (NO secrets here)
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP")
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME")

missing = [name for name, val in [
    ("AZURE_SUBSCRIPTION_ID", SUBSCRIPTION_ID),
    ("AZURE_RESOURCE_GROUP", RESOURCE_GROUP),
    ("AZUREML_WORKSPACE_NAME", WORKSPACE_NAME),
] if not val]
if missing:
    raise EnvironmentError(f"Missing required env vars: {', '.join(missing)}")
SUBSCRIPTION_ID_GPU = os.getenv("AZURE_SUBSCRIPTION_ID_GPU")
RESOURCE_GROUP_GPU = os.getenv("AZURE_RESOURCE_GROUP_GPU")
WORKSPACE_NAME_GPU = os.getenv("AZUREML_WORKSPACE_NAME_GPU")
missing_gpu = [name for name, val in [
    ("AZURE_SUBSCRIPTION_ID_GPU", SUBSCRIPTION_ID_GPU),
    ("AZURE_RESOURCE_GROUP_GPU", RESOURCE_GROUP_GPU),
    ("AZUREML_WORKSPACE_NAME_GPU", WORKSPACE_NAME_GPU),
] if not val]
if missing_gpu:
    raise EnvironmentError(f"Missing required env vars: {', '.join(missing_gpu)}")


# DeepFace Configuration
DF_ENDPOINT_NAME = "deep-face-endpoint-v2"
DF_MODEL_NAME = "deepface-models"
DF_MODEL_VERSION = "1"
DF_ENV_NAME = "deepface-online-endpoint-environment"
DF_ENV_VERSION = "1"
DF_DEPLOYMENT_NAME = "blue"

# GroundingDino Configuration
GD_ENDPOINT_NAME = "grounding-dino-endpoint"
GD_MODEL_NAME = "groundingdino-model"
GD_MODEL_VERSION = "1"
GD_ENV_NAME = "groundingdino-online-endpoint-environment"
GD_ENV_VERSION = "3"
GD_DEPLOYMENT_NAME = "blue"

# CLIP Configuration
CLIP_ENDPOINT_NAME = "clip-endpoint"
CLIP_MODEL_NAME = "clip-vit-large-patch14"
CLIP_MODEL_VERSION = "1"
CLIP_ENV_NAME = "clip-online-endpoint-environment"
CLIP_ENV_VERSION = "1"
CLIP_DEPLOYMENT_NAME = "blue"

# Combined CLIP + GroundingDINO Configuration
CG_ENDPOINT_NAME = "clip-groundingdino-endpoint"
CG_MODEL_NAME = "clip-groundingdino-model"
CG_MODEL_VERSION = "1"
CG_ENV_NAME = "clip-groundingdino-online-endpoint-environment"
CG_ENV_VERSION = "14"
CG_DEPLOYMENT_NAME = "blue"

# Local folders in this repo
THIS_DIR = Path.cwd()
SCORING_CODE_DIR = THIS_DIR / "DeepFace" / "ScoringScript"
CONDA_FILE = THIS_DIR / "DeepFace" / "Environment" / "conda.yaml"

GD_SCORING_CODE_DIR = THIS_DIR / "GroundingDino" / "ScoringScript"
GD_CONDA_FILE = THIS_DIR / "GroundingDino" / "Environment" / "conda.yaml"

CLIP_SCORING_CODE_DIR = THIS_DIR / "CLIP" / "ScoringScript"
CLIP_CONDA_FILE = THIS_DIR / "CLIP" / "Environment" / "conda.yaml"

CG_SCORING_CODE_DIR = THIS_DIR / "CLIP_GroundingDino" / "ScoringScript"
CG_CONDA_FILE = THIS_DIR / "CLIP_GroundingDino" / "Environment" / "conda.yaml"

# Local folders contain model cache
CACHE_DIR = THIS_DIR.parent.parent / "temporary" / "cache"

# 2) Connect to Azure ML Workspace
We use `DefaultAzureCredential` so secrets are never hardcoded in the notebook.

In [3]:
gpu_credential = ClientSecretCredential(
    tenant_id=TENANT_ID_GPU,
    client_id=CLIENT_ID_GPU,
    client_secret=CLIENT_SECRET_GPU
) 

ml_client_with_gpu = MLClient(
    credential=gpu_credential,
    subscription_id=SUBSCRIPTION_ID_GPU,
    resource_group_name=RESOURCE_GROUP_GPU,
    workspace_name=WORKSPACE_NAME_GPU,
)
print("Connected to workspace:", WORKSPACE_NAME_GPU)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Connected to workspace: SoccerAgent


In [4]:
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
) 

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print("Connected to workspace:", WORKSPACE_NAME)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected to workspace: SoccerAgentWS


# 3) Get-or-create  model asset


In [5]:
# Get-or-create the DeepFace weights model asset

def try_get_model(name: str, version: str, ml_client: MLClient) -> Model:
    try:
        return ml_client.models.get(name=name, version=version)
    except ResourceNotFoundError:
        return None

def ensure_deepface_model(ml_client: MLClient, using_local_model=False) -> Model:
    # If using_local_model is True, we skip checking existing models in the workspace
    if not using_local_model:
        existing = try_get_model(DF_MODEL_NAME, DF_MODEL_VERSION, ml_client)
        if existing is not None:
            print(f"Model exists: {DF_MODEL_NAME}:{DF_MODEL_VERSION}")
            return existing

        print(f"Model not found. Creating: {DF_MODEL_NAME}:{DF_MODEL_VERSION}")
    
    # Download DeepFace model weights
    tmp_root = CACHE_DIR / ".deepface" / "weights"
    tmp_root.mkdir(parents=True, exist_ok=True)

    facenet_url = "https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.h5"
    retinaface_url = "https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5"

    facenet_path = tmp_root / "facenet512_weights.h5"
    retinaface_path = tmp_root / "retinaface.h5"

    if not facenet_path.exists():
        urllib.request.urlretrieve(facenet_url, str(facenet_path))
    if not retinaface_path.exists():
        urllib.request.urlretrieve(retinaface_url, str(retinaface_path))

    # Create the model asset
    model_asset = Model(
        name=DF_MODEL_NAME,
        version=DF_MODEL_VERSION,
        path=str((CACHE_DIR / ".deepface")),
        description="DeepFace model weights folder (.deepface/weights)",
    )

    # Return the model asset directly if using local model
    if using_local_model:
        return model_asset
    
    # Register the model in the workspace
    created = ml_client.models.create_or_update(model_asset)
    print(f"Created model: {created.name}:{created.version}")

    return created

def ensure_groundingdino_model(ml_client: MLClient, using_local_model=False) -> Model:
    if not using_local_model:
        existing = try_get_model(GD_MODEL_NAME, GD_MODEL_VERSION, ml_client)
        if existing is not None:
            print(f"Model exists: {GD_MODEL_NAME}:{GD_MODEL_VERSION}")
            return existing

        print(f"Model not found. Creating: {GD_MODEL_NAME}:{GD_MODEL_VERSION}")
    
    # Download model from HuggingFace
    tmp_root = CACHE_DIR / ".groundingdino"
    tmp_root.mkdir(parents=True, exist_ok=True)
    
    print("Downloading GroundingDino model from HuggingFace...")
    #snapshot_download(repo_id="IDEA-Research/grounding-dino-base", local_dir=tmp_root)
    
    model_asset = Model(
        name=GD_MODEL_NAME,
        version=GD_MODEL_VERSION,
        path=str(tmp_root),
        description="GroundingDINO model weights (IDEA-Research/grounding-dino-base)",
    )
    if using_local_model:
        return model_asset
    
    created = ml_client.models.create_or_update(model_asset)
    print(f"Created model: {created.name}:{created.version}")
    return created

def ensure_clip_model(ml_client: MLClient, using_local_model=False) -> Model:
    if not using_local_model:
        existing = try_get_model(CLIP_MODEL_NAME, CLIP_MODEL_VERSION, ml_client)
        if existing is not None:
            print(f"Model exists: {CLIP_MODEL_NAME}:{CLIP_MODEL_VERSION}")
            return existing

        print(f"Model not found. Creating: {CLIP_MODEL_NAME}:{CLIP_MODEL_VERSION}")

    tmp_root = CACHE_DIR / ".clip"
    tmp_root.mkdir(parents=True, exist_ok=True)

    print("Downloading CLIP model from HuggingFace (openai/clip-vit-large-patch14)...")
    #snapshot_download(repo_id="openai/clip-vit-large-patch14", local_dir=tmp_root)
    
    model_asset = Model(
        name=CLIP_MODEL_NAME,
        version=CLIP_MODEL_VERSION,
        path=str(tmp_root),
        description="CLIP ViT-L/14 model weights (openai/clip-vit-large-patch14)",
    )
    if using_local_model:
        return model_asset

    created = ml_client.models.create_or_update(model_asset)
    print(f"Created model: {created.name}:{created.version}")
    return created

def ensure_clip_groundingdino_model(ml_client: MLClient, using_local_model: bool = False) -> Model:
    if not using_local_model:
        existing = try_get_model(CG_MODEL_NAME, CG_MODEL_VERSION, ml_client)
        if existing is not None:
            print(f"Model exists: {CG_MODEL_NAME}:{CG_MODEL_VERSION}")
            return existing
        print(f"Model not found. Creating: {CG_MODEL_NAME}:{CG_MODEL_VERSION}")
    
    combo_root = CACHE_DIR / ".groundingdino_and_clip"
    clip_dir = combo_root / ".clip"
    gd_dir = combo_root / ".groundingdino"
    clip_dir.mkdir(parents=True, exist_ok=True)
    gd_dir.mkdir(parents=True, exist_ok=True)
    
    if not any(clip_dir.iterdir()):
        print("Downloading CLIP model from HuggingFace (openai/clip-vit-large-patch14)...")
        snapshot_download(
            repo_id="openai/clip-vit-large-patch14",
            local_dir=clip_dir,
            local_dir_use_symlinks=False,
        )
    else:
        print("CLIP cache found; skipping download")
    
    if not any(gd_dir.iterdir()):
        print("Downloading GroundingDino model from HuggingFace (IDEA-Research/grounding-dino-base)...")
        snapshot_download(
            repo_id="IDEA-Research/grounding-dino-base",
            local_dir=gd_dir,
            local_dir_use_symlinks=False,
        )
    else:
        print("GroundingDino cache found; skipping download")
    
    model_asset = Model(
        name=CG_MODEL_NAME,
        version=CG_MODEL_VERSION,
        path=str(combo_root),
        description="CLIP + GroundingDINO combined model weights",
    )
    if using_local_model:
        return model_asset
    
    created = ml_client.models.create_or_update(model_asset)
    print(f"Created model: {created.name}:{created.version}")
    return created

deepface_model = ensure_deepface_model(ml_client, using_local_model=False)
#groundingdino_model = ensure_groundingdino_model(ml_client, using_local_model=False)
#clip_model = ensure_clip_model(ml_client, using_local_model=False)
clip_groundingdino_model = ensure_clip_groundingdino_model(ml_client_with_gpu, using_local_model=False)

Model exists: deepface-models:1
Model exists: clip-groundingdino-model:1


# 4) Get-or-create the environment
This environment must contain your runtime dependencies (from `DeepFaceOnlineScoring/Environment/conda.yaml`).

In [6]:
def try_get_environment(name: str, version: str, ml_client: MLClient = ml_client) -> Environment:
    try:
        return ml_client.environments.get(name=name, version=version)
    except ResourceNotFoundError:
        return None

def ensure_deepface_environment(ml_client: MLClient) -> Environment:
    existing = try_get_environment(DF_ENV_NAME, DF_ENV_VERSION, ml_client)
    if existing is not None:
        print(f"Environment exists: {DF_ENV_NAME}:{DF_ENV_VERSION}")
        return existing

    print(f"Environment not found. Creating: {DF_ENV_NAME}:{DF_ENV_VERSION}")
    env = Environment(
        name=DF_ENV_NAME,
        version=DF_ENV_VERSION,
        description="Environment for DeepFace online endpoint",
        conda_file=str(CONDA_FILE),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    )
    created = ml_client.environments.create_or_update(env)
    print(f"Created environment: {created.name}:{created.version}")
    return created

def ensure_groundingdino_environment(ml_client: MLClient) -> Environment:
    existing = try_get_environment(GD_ENV_NAME, GD_ENV_VERSION, ml_client)
    if existing is not None:
        print(f"Environment exists: {GD_ENV_NAME}:{GD_ENV_VERSION}")
        return existing

    print(f"Environment not found. Creating: {GD_ENV_NAME}:{GD_ENV_VERSION}")
    env = Environment(
        name=GD_ENV_NAME,
        version=GD_ENV_VERSION,
        description="Environment for GroundingDino online endpoint",
        conda_file=str(GD_CONDA_FILE),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    )
    created = ml_client.environments.create_or_update(env)
    print(f"Created environment: {created.name}:{created.version}")
    return created

def ensure_clip_environment(ml_client: MLClient) -> Environment:
    existing = try_get_environment(CLIP_ENV_NAME, CLIP_ENV_VERSION, ml_client)
    if existing is not None:
        print(f"Environment exists: {CLIP_ENV_NAME}:{CLIP_ENV_VERSION}")
        return existing

    print(f"Environment not found. Creating: {CLIP_ENV_NAME}:{CLIP_ENV_VERSION}")
    env = Environment(
        name=CLIP_ENV_NAME,
        version=CLIP_ENV_VERSION,
        description="Environment for CLIP online endpoint",
        conda_file=str(CLIP_CONDA_FILE),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    )
    created = ml_client.environments.create_or_update(env)
    print(f"Created environment: {created.name}:{created.version}")
    return created

def ensure_clip_groundingdino_environment(ml_client: MLClient) -> Environment:
    existing = try_get_environment(CG_ENV_NAME, CG_ENV_VERSION, ml_client)
    if existing is not None:
        print(f"Environment exists: {CG_ENV_NAME}:{CG_ENV_VERSION}")
        return existing
    
    print(f"Environment not found. Creating: {CG_ENV_NAME}:{CG_ENV_VERSION}")
    env = Environment(
        name=CG_ENV_NAME,
        version=CG_ENV_VERSION,
        description="Environment for CLIP + GroundingDINO combined endpoint",
        image="mcr.microsoft.com/azureml/curated/acpt-pytorch-2.2-cuda12.1:45",
        conda_file=str(CG_CONDA_FILE),
    )
    created = ml_client.environments.create_or_update(env)
    print(f"Created environment: {created.name}:{created.version}")
    return created

deepface_env = ensure_deepface_environment(ml_client)
# groundingdino_env = ensure_groundingdino_environment(ml_client)
# clip_env = ensure_clip_environment(ml_client)
clip_groundingdino_env = ensure_clip_groundingdino_environment(ml_client_with_gpu)

Environment exists: deepface-online-endpoint-environment:1
Environment exists: clip-groundingdino-online-endpoint-environment:14


# 5) Get-or-create the endpoint


In [7]:
def try_get_endpoint(name: str, ml_client: MLClient) -> ManagedOnlineEndpoint:
    try:
        return ml_client.online_endpoints.get(name=name)
    except ResourceNotFoundError:
        return None

def ensure_deepface_endpoint(ml_client: MLClient, deploy_local=False):
    # If deploy_local is True, we skip checking existing endpoints in the workspace
    if not deploy_local:
        existing = try_get_endpoint(DF_ENDPOINT_NAME, ml_client)
        if existing is not None:
            print(f"Endpoint exists: {DF_ENDPOINT_NAME}")
            return existing

        print(f"Endpoint not found. Creating: {DF_ENDPOINT_NAME}")

    # Create the endpoint
    endpoint = ManagedOnlineEndpoint(
        name=DF_ENDPOINT_NAME,
        description="DeepFace scoring endpoint",
        auth_mode="key",
    )
    return  ml_client.online_endpoints.begin_create_or_update(endpoint, local=deploy_local)

def ensure_groundingdino_endpoint(ml_client: MLClient, deploy_local=False):
    # If deploy_local is True, we skip checking existing endpoints in the workspace
    if not deploy_local:
        existing = try_get_endpoint(GD_ENDPOINT_NAME, ml_client)
        if existing is not None:
            print(f"Endpoint exists: {GD_ENDPOINT_NAME}")
            return existing
        print(f"Endpoint not found. Creating: {GD_ENDPOINT_NAME}")

    # Create the endpoint
    endpoint = ManagedOnlineEndpoint(
        name=GD_ENDPOINT_NAME,
        description="GroundingDino scoring endpoint",
        auth_mode="key",
    )
    return ml_client.online_endpoints.begin_create_or_update(endpoint, local=deploy_local)

def ensure_clip_endpoint(ml_client: MLClient, deploy_local=False):
    if not deploy_local:
        existing = try_get_endpoint(CLIP_ENDPOINT_NAME, ml_client)
        if existing is not None:
            print(f"Endpoint exists: {CLIP_ENDPOINT_NAME}")
            return existing
        print(f"Endpoint not found. Creating: {CLIP_ENDPOINT_NAME}")

    endpoint = ManagedOnlineEndpoint(
        name=CLIP_ENDPOINT_NAME,
        description="CLIP frame selection endpoint",
        auth_mode="key",
    )
    return ml_client.online_endpoints.begin_create_or_update(endpoint, local=deploy_local)

def ensure_clip_groundingdino_endpoint(ml_client: MLClient, deploy_local=False):
    if not deploy_local:
        existing = try_get_endpoint(CG_ENDPOINT_NAME, ml_client)
        if existing is not None:
            print(f"Endpoint exists: {CG_ENDPOINT_NAME}")
            return existing
        print(f"Endpoint not found. Creating: {CG_ENDPOINT_NAME}")

    endpoint = ManagedOnlineEndpoint(
        name=CG_ENDPOINT_NAME,
        description="CLIP + GroundingDINO combined scoring endpoint",
        auth_mode="key",
    )
    return ml_client.online_endpoints.begin_create_or_update(endpoint, local=deploy_local)

# Start endpoint creation in parallel
df_endpoint_poller = ensure_deepface_endpoint(ml_client, deploy_local=False)
# gd_endpoint_poller = ensure_groundingdino_endpoint(ml_client, deploy_local=False)
# clip_endpoint_poller = ensure_clip_endpoint(ml_client, deploy_local=False)
cg_endpoint_poller = ensure_clip_groundingdino_endpoint(ml_client_with_gpu, deploy_local=False)

# Wait for all to complete

if isinstance(df_endpoint_poller, LROPoller):
    df_endpoint = df_endpoint_poller.result()
    print(f"Created endpoint: {df_endpoint.name}")
else:
    df_endpoint = df_endpoint_poller

# if isinstance(gd_endpoint_poller, LROPoller):
#     gd_endpoint = gd_endpoint_poller.result()
#     print(f"Created endpoint: {gd_endpoint.name}")
# else:
#     gd_endpoint = gd_endpoint_poller

# if isinstance(clip_endpoint_poller, LROPoller):
#     clip_endpoint = clip_endpoint_poller.result()
#     print(f"Created endpoint: {clip_endpoint.name}")
# else:
#     clip_endpoint = clip_endpoint_poller

if isinstance(cg_endpoint_poller, LROPoller):
    cg_endpoint = cg_endpoint_poller.result()
    print(f"Created endpoint: {cg_endpoint.name}")
else:
    cg_endpoint = cg_endpoint_poller

Endpoint exists: deep-face-endpoint-v2
Endpoint exists: clip-groundingdino-endpoint


# 6) Get-or-create the deployment
We create a single deployment named `blue` and route 100% traffic to it.

In [8]:
def try_get_deployment(endpoint_name: str, deployment_name: str, ml_client: MLClient) -> ManagedOnlineDeployment:
    try:
        return ml_client.online_deployments.get(name=deployment_name, endpoint_name=endpoint_name)
    except ResourceNotFoundError:
        return None

def ensure_deepface_deployment(ml_client: MLClient, deploy_local=False):
    #If deploy endpoint to local, we skip checking existing deployments in the workspace
    if not deploy_local:
        existing = try_get_deployment(DF_ENDPOINT_NAME, DF_DEPLOYMENT_NAME, ml_client)
        if existing is not None:
            print(f"Deployment exists: {DF_DEPLOYMENT_NAME} (endpoint={DF_ENDPOINT_NAME})")
            return existing
        print(f"Deployment not found. Creating: {DF_DEPLOYMENT_NAME} (endpoint={DF_ENDPOINT_NAME})")
    
    # Create the deployment
    deployment = ManagedOnlineDeployment(
        name=DF_DEPLOYMENT_NAME,
        endpoint_name=DF_ENDPOINT_NAME,
        model=deepface_model,
        environment=deepface_env,
        code_configuration=CodeConfiguration(
            code=str(SCORING_CODE_DIR),
            scoring_script="score.py",
        ),
        instance_type="Standard_F4s_v2",
        instance_count=1,
        request_settings=OnlineRequestSettings(
            max_concurrent_requests_per_instance=1,
            request_timeout_ms=60000,
        ),
        app_insights_enabled=True

    )
    return ml_client.online_deployments.begin_create_or_update(deployment, local=deploy_local)

def ensure_groundingdino_deployment(ml_client: MLClient, deploy_local=False):
    if not deploy_local:
        existing = try_get_deployment(GD_ENDPOINT_NAME, GD_DEPLOYMENT_NAME, ml_client)
        if existing is not None:
            print(f"Deployment exists: {GD_DEPLOYMENT_NAME} (endpoint={GD_ENDPOINT_NAME})")
            return existing
        print(f"Deployment not found. Creating: {GD_DEPLOYMENT_NAME} (endpoint={GD_ENDPOINT_NAME})")

    # Create the deployment
    deployment = ManagedOnlineDeployment(
        name=GD_DEPLOYMENT_NAME,
        endpoint_name=GD_ENDPOINT_NAME,
        model=groundingdino_model,
        environment=groundingdino_env,
        code_configuration=CodeConfiguration(
            code=str(GD_SCORING_CODE_DIR),
            scoring_script="score.py",
        ),
        instance_type="Standard_F4s_v2", # GPU instance for GroundingDino Standard_NC4as_T4_v3
        instance_count=1,
        request_settings=OnlineRequestSettings(
            max_concurrent_requests_per_instance=1,
            request_timeout_ms=60000,
        ),
        app_insights_enabled=True
    )
    return ml_client.online_deployments.begin_create_or_update(deployment, local=deploy_local)

def ensure_clip_deployment(ml_client: MLClient, deploy_local=False):
    if not deploy_local:
        existing = try_get_deployment(CLIP_ENDPOINT_NAME, CLIP_DEPLOYMENT_NAME, ml_client)
        if existing is not None:
            print(f"Deployment exists: {CLIP_DEPLOYMENT_NAME} (endpoint={CLIP_ENDPOINT_NAME})")
            return existing
        print(f"Deployment not found. Creating: {CLIP_DEPLOYMENT_NAME} (endpoint={CLIP_ENDPOINT_NAME})")

    deployment = ManagedOnlineDeployment(
        name=CLIP_DEPLOYMENT_NAME,
        endpoint_name=CLIP_ENDPOINT_NAME,
        model=clip_model,
        environment=clip_env,
        code_configuration=CodeConfiguration(
            code=str(CLIP_SCORING_CODE_DIR),
            scoring_script="score.py",
        ),
        instance_type="Standard_F4s_v2",
        instance_count=1,
        request_settings=OnlineRequestSettings(
            max_concurrent_requests_per_instance=1, # GPU instance for GroundingDino Standard_NC4as_T4_v3
            request_timeout_ms=45000,
        ),
        app_insights_enabled=True
    )
    return ml_client.online_deployments.begin_create_or_update(deployment, local=deploy_local)

def ensure_clip_groundingdino_deployment(ml_client: MLClient, deploy_local: bool = False):
    if not deploy_local:
        existing = try_get_deployment(CG_ENDPOINT_NAME, CG_DEPLOYMENT_NAME, ml_client)
        if existing is not None:
            print(f"Deployment exists: {CG_DEPLOYMENT_NAME} (endpoint={CG_ENDPOINT_NAME})")
            return existing
        print(f"Deployment not found. Creating: {CG_DEPLOYMENT_NAME} (endpoint={CG_ENDPOINT_NAME})")
    
    deployment = ManagedOnlineDeployment(
        name=CG_DEPLOYMENT_NAME,
        endpoint_name=CG_ENDPOINT_NAME,
        model=clip_groundingdino_model,
        environment=clip_groundingdino_env,
        code_configuration=CodeConfiguration(
            code=str(CG_SCORING_CODE_DIR),
            scoring_script="score.py",
        ),
        instance_type="Standard_NC4as_T4_v3",
        instance_count=1,
        request_settings=OnlineRequestSettings(
            max_concurrent_requests_per_instance=1,
            request_timeout_ms=60000,
        ),
        app_insights_enabled=True
    )
    return ml_client.online_deployments.begin_create_or_update(deployment, local=deploy_local)

# Start deployment creation in parallel
print("Starting deployments in parallel...")
df_deployment_poller = ensure_deepface_deployment(ml_client, deploy_local=False)
# gd_deployment_poller = ensure_groundingdino_deployment(ml_client, deploy_local=False)
# clip_deployment_poller = ensure_clip_deployment(ml_client, deploy_local=False)
cg_deployment_poller = ensure_clip_groundingdino_deployment(ml_client_with_gpu, deploy_local=False)

# Wait for all to complete
if isinstance(df_deployment_poller, LROPoller):
    print(f"Waiting for DeepFace deployment '{DF_DEPLOYMENT_NAME}'...")
    df_deployment = df_deployment_poller.result()
    print(f"Created deployment: {df_deployment.name}")
else:
    df_deployment = df_deployment_poller

# if isinstance(gd_deployment_poller, LROPoller):
#     print(f"Waiting for GroundingDino deployment '{GD_DEPLOYMENT_NAME}'...")
#     gd_deployment = gd_deployment_poller.result()
#     print(f"Created deployment: {gd_deployment.name}")
# else:
#     gd_deployment = gd_deployment_poller

# if isinstance(clip_deployment_poller, LROPoller):
#     print(f"Waiting for CLIP deployment '{CLIP_DEPLOYMENT_NAME}'...")
#     clip_deployment = clip_deployment_poller.result()
#     print(f"Created deployment: {clip_deployment.name}")
# else:
#     clip_deployment = clip_deployment_poller

if isinstance(cg_deployment_poller, LROPoller):
    print(f"Waiting for CLIP+GroundingDINO deployment '{CG_DEPLOYMENT_NAME}'...")
    cg_deployment = cg_deployment_poller.result()
    print(f"Created deployment: {cg_deployment.name}")
else:
    cg_deployment = cg_deployment_poller

Starting deployments in parallel...


Check: endpoint deep-face-endpoint-v2 exists


Deployment not found. Creating: blue (endpoint=deep-face-endpoint-v2)


Uploading ScoringScript (0.01 MBs): 100%|##########| 8983/8983 [00:00<00:00, 32609.92it/s]




.Deployment exists: blue (endpoint=clip-groundingdino-endpoint)
Waiting for DeepFace deployment 'blue'...
..............................................................................Created deployment: blue


In [9]:
# # Route 100% of traffic to the deployment (safe to re-run)
df_endpoint = ml_client.online_endpoints.get(name=DF_ENDPOINT_NAME)
df_endpoint.traffic = {DF_DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(df_endpoint)
print(f"Traffic for {DF_ENDPOINT_NAME} set to", df_endpoint.traffic)

# gd_endpoint = ml_client.online_endpoints.get(name=GD_ENDPOINT_NAME)
# gd_endpoint.traffic = {GD_DEPLOYMENT_NAME: 100}
# ml_client.online_endpoints.begin_create_or_update(gd_endpoint).result()
# print(f"Traffic for {GD_ENDPOINT_NAME} set to", gd_endpoint.traffic)

# clip_endpoint = ml_client.online_endpoints.get(name=CLIP_ENDPOINT_NAME)
# clip_endpoint.traffic = {CLIP_DEPLOYMENT_NAME: 100}
# ml_client.online_endpoints.begin_create_or_update(clip_endpoint).result()
# print(f"Traffic for {CLIP_ENDPOINT_NAME} set to", clip_endpoint.traffic)

cg_endpoint = ml_client_with_gpu.online_endpoints.get(name=CG_ENDPOINT_NAME)
cg_endpoint.traffic = {CG_DEPLOYMENT_NAME: 100}
ml_client_with_gpu.online_endpoints.begin_create_or_update(cg_endpoint).result()
print(f"Traffic for {CG_ENDPOINT_NAME} set to", cg_endpoint.traffic)

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Traffic for deep-face-endpoint-v2 set to {'blue': 100}


Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Traffic for clip-groundingdino-endpoint set to {'blue': 100}


# 7) Get scoring URI and API key (Azure ML SDK)
This is the canonical way to fetch the endpoint URL and secret without hardcoding.

In [10]:
df_endpoint = ml_client.online_endpoints.get(name=DF_ENDPOINT_NAME)
df_keys = ml_client.online_endpoints.get_keys(name=DF_ENDPOINT_NAME)

# print("=== DeepFace Endpoint ===")
# print("Endpoint name:", df_endpoint.name)
# print("Scoring URI:", df_endpoint.scoring_uri)
# print("Primary key (secret):", df_keys.primary_key)
# print()

# gd_endpoint = ml_client.online_endpoints.get(name=GD_ENDPOINT_NAME)
# gd_keys = ml_client.online_endpoints.get_keys(name=GD_ENDPOINT_NAME)

# print("=== GroundingDino Endpoint ===")
# print("Endpoint name:", gd_endpoint.name)
# print("Scoring URI:", gd_endpoint.scoring_uri)
# print("Primary key (secret):", gd_keys.primary_key)
# print()

# clip_endpoint = ml_client.online_endpoints.get(name=CLIP_ENDPOINT_NAME)
# clip_keys = ml_client.online_endpoints.get_keys(name=CLIP_ENDPOINT_NAME)

# print("=== CLIP Endpoint ===")
# print("Endpoint name:", clip_endpoint.name)
# print("Scoring URI:", clip_endpoint.scoring_uri)
# print("Primary key (secret):", clip_keys.primary_key)
# print()

cg_endpoint = ml_client_with_gpu.online_endpoints.get(name=CG_ENDPOINT_NAME)
cg_keys = ml_client_with_gpu.online_endpoints.get_keys(name=CG_ENDPOINT_NAME)

# Save these values to dotenv
dotenv_path = Path(find_dotenv())
set_key(dotenv_path, "DEEPFACE_ENDPOINT_URI", df_endpoint.scoring_uri) if df_endpoint.scoring_uri else None
set_key(dotenv_path, "DEEPFACE_ENDPOINT_KEY", df_keys.primary_key) if df_keys.primary_key else None
# set_key(dotenv_path, "GROUNDINGDINO_ENDPOINT_URI", gd_endpoint.scoring_uri) if gd_endpoint.scoring_uri else None
# set_key(dotenv_path, "GROUNDINGDINO_ENDPOINT_KEY", gd_keys.primary_key) if gd_keys.primary_key else None
# set_key(dotenv_path, "CLIP_ENDPOINT_URI", clip_endpoint.scoring_uri) if clip_endpoint.scoring_uri else None
# set_key(dotenv_path, "CLIP_ENDPOINT_KEY", clip_keys.primary_key) if clip_keys.primary_key else None
set_key(dotenv_path, "CLIP_GROUNDINGDINO_ENDPOINT_URI", cg_endpoint.scoring_uri) if cg_endpoint.scoring_uri else None
set_key(dotenv_path, "CLIP_GROUNDINGDINO_ENDPOINT_KEY", cg_keys.primary_key) if cg_keys.primary_key else None

(True,
 'CLIP_GROUNDINGDINO_ENDPOINT_KEY',
 '90EiF3ALci5M93KMytYl22FeNWVAUdozsNzc92q9TK8Fty8qXHRtJQQJ99CAAAAAAAAAAAAAINFRAZML4Gy4')

# 8) Invoke the endpoint
Choose either SDK invocation (recommended) or direct REST call (useful outside AML).

## 8.1 Create a sample request JSON
Set `IMAGE_PATH` to a local image file. This creates `sample_request.json` with a base64-encoded image.

In [11]:
# IMAGE_PATH = "E:\\Repository\\FinalProject\\scripts\\evaluation\\SoccerBench_dataset\\materials\\q4\\pic\\0bDEBms3\\3.jpg"  # optional env var to avoid editing the notebook
# if IMAGE_PATH:
#     image_path = Path(IMAGE_PATH)
#     if not image_path.exists():
#         raise FileNotFoundError(f"Sample image not found: {image_path}")
    
#     with open(image_path, "rb") as f:
#         image_base64 = base64.b64encode(f.read()).decode("utf-8")
#     payload = {"image": image_base64,
#                "task": "groundingdino",
#                "text_threshold": 0.3,
#                 "threshold": 0.2,
#                "queries": ["a person wearing white shirt.a player wearing white shirt"]
#     }
#     with open(THIS_DIR / "sample_request.json", "w", encoding="utf-8") as f:
#         json.dump(payload, f)
#     print("Wrote sample_request.json")

## 8.2 Invoke via Azure ML SDK

In [12]:
# request_file = str(THIS_DIR / "sample_request.json")
# if not Path(request_file).exists():
#     raise FileNotFoundError(f"Request file not found: {request_file}")
    
# result = ml_client_with_gpu.online_endpoints.invoke(
#     endpoint_name=DF_ENDPOINT_NAME,
#     deployment_name=DF_DEPLOYMENT_NAME,
#     request_file=request_file,
# )
# print(result)

## 8.3 Invoke via REST (outside Azure ML)
This uses the scoring URI + primary key retrieved above.

In [13]:
# import requests

# endpoint = ml_client_with_gpu.online_endpoints.get(name=CG_ENDPOINT_NAME)
# keys = ml_client_with_gpu.online_endpoints.get_keys(name=CG_ENDPOINT_NAME)

# scoring_uri = endpoint.scoring_uri
# api_key = keys.primary_key

# with open(THIS_DIR / "sample_request.json", "r", encoding="utf-8") as f:
#     payload = json.load(f)

# headers = {
#     "Content-Type": "application/json",
#     "Authorization": f"Bearer {api_key}",
# }
# resp = requests.post(url=scoring_uri, json=payload, headers=headers, timeout=60)
# print("Status:", resp.status_code)
# print(resp.text)

# 9) Delete the endpoint (cleanup)
Deleting the endpoint stops billing for the managed online endpoint + deployment.

**Safety:** the code below is guarded; you must explicitly set `CONFIRM_DELETE=True` for it to run.

In [14]:
# DANGER ZONE: Cleanup / delete endpoint
from azure.core.exceptions import ResourceNotFoundError

# Set this to True to actually delete resources.
CONFIRM_DELETE = False

if not CONFIRM_DELETE:
    raise RuntimeError(
        f"Refusing to delete endpoints. Set CONFIRM_DELETE=True to proceed."
    )

# Delete DeepFace Endpoint Deployment
try:
    print(f"Routeing 0% traffic to deployment '{DF_DEPLOYMENT_NAME}' before deletion...")
    endpoint = ml_client.online_endpoints.get(name=DF_ENDPOINT_NAME)
    endpoint.traffic = {DF_DEPLOYMENT_NAME: 0}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()

    print(f"Deleting deployment '{DF_DEPLOYMENT_NAME}' from endpoint '{DF_ENDPOINT_NAME}'...")
    ml_client.online_deployments.begin_delete(
        name=DF_DEPLOYMENT_NAME,
        endpoint_name=DF_ENDPOINT_NAME,
    ).result()
    print("Deployment deleted.")
    
except ResourceNotFoundError:
    print(f"DeepFace resources not found (already deleted).")

# # Delete GroundingDino Endpoint Deployment
# try:
#     print(f"Routeing 0% traffic to deployment '{GD_DEPLOYMENT_NAME}' before deletion...")
#     gd_endpoint = ml_client.online_endpoints.get(name=GD_ENDPOINT_NAME)
#     gd_endpoint.traffic = {GD_DEPLOYMENT_NAME: 0}
#     ml_client.online_endpoints.begin_create_or_update(gd_endpoint).result()

#     print(f"Deleting deployment '{GD_DEPLOYMENT_NAME}' from endpoint '{GD_ENDPOINT_NAME}'...")
#     ml_client.online_deployments.begin_delete(
#         name=GD_DEPLOYMENT_NAME,
#         endpoint_name=GD_ENDPOINT_NAME,
#     ).result()
#     print("Deployment deleted.")

# except ResourceNotFoundError:
#     print(f"GroundingDino resources not found (already deleted).")

# # Delete CLIP Endpoint
# try:
#     print(f"Routeing 0% traffic to deployment '{CLIP_DEPLOYMENT_NAME}' before deletion...")
#     clip_endpoint = ml_client.online_endpoints.get(name=CLIP_ENDPOINT_NAME)
#     clip_endpoint.traffic = {CLIP_DEPLOYMENT_NAME: 0}
#     ml_client.online_endpoints.begin_create_or_update(clip_endpoint).result()

#     print(f"Deleting deployment '{CLIP_DEPLOYMENT_NAME}' from endpoint '{CLIP_ENDPOINT_NAME}'...")
#     ml_client.online_deployments.begin_delete(
#         name=CLIP_DEPLOYMENT_NAME,
#         endpoint_name=CLIP_ENDPOINT_NAME,
#     ).result()
#     print("Deployment deleted.")
# except ResourceNotFoundError:
#     print(f"CLIP resources not found (already deleted).")

# Delete CLIP + GroundingDINO Endpoint
try:
    print(f"Routeing 0% traffic to deployment '{CG_DEPLOYMENT_NAME}' before deletion...")
    cg_endpoint_poller.traffic = {CG_DEPLOYMENT_NAME: 0}
    ml_client_with_gpu.online_endpoints.begin_create_or_update(cg_endpoint_poller).result()

    print(f"Deleting deployment '{CG_DEPLOYMENT_NAME}' from endpoint '{CG_ENDPOINT_NAME}'...")
    ml_client_with_gpu.online_deployments.begin_delete(
        name=CG_DEPLOYMENT_NAME,
        endpoint_name=CG_ENDPOINT_NAME,
    ).result()
    print("Deployment deleted.")
except ResourceNotFoundError:
    print(f"CLIP + GroundingDINO resources not found (already deleted).")

RuntimeError: Refusing to delete endpoints. Set CONFIRM_DELETE=True to proceed.